In [1]:
!pip install langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Cargar el texto que guardamos en el paso anterior
ruta_txt = '/content/drive/MyDrive/u/cd2/texto_banco_atlas.txt'
with open(ruta_txt, 'r', encoding='utf-8') as f:
    texto_completo = f.read()

# --- JUSTIFICACIÓN PARA LA RÚBRICA ---
# Chunk Size = 1000: Es un tamaño ideal para manuales corporativos. Permite que cada bloque
# contenga un artículo o política completa (aprox. 150-200 palabras) sin diluir el tema principal.
# Chunk Overlap = 200: Asegura que si una regla del banco se corta al final de un chunk,
# el siguiente bloque herede las últimas palabras para no perder el contexto (solapamiento).

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

# 3. Dividir el texto
chunks = text_splitter.split_text(texto_completo)

print(f"✅ El documento se ha dividido en {len(chunks)} fragmentos (chunks).")
print("\n--- Vista previa del Chunk 0 ---")
print(chunks[0])
print("\n--- Vista previa del Chunk 1 ---")
print(chunks[1])

✅ El documento se ha dividido en 238 fragmentos (chunks).

--- Vista previa del Chunk 0 ---
BANCO ATLAS FINANCIERO S.A.
Manual Corporativo Consolidado y Políticas Operativas
Edición Extendida 2026. Este documento contiene la totalidad de las normativas de
Recursos Humanos, Reglamento Interno de Trabajo, Compliance, Seguridad de la
Información, Beneficios, Relación de Puestos de Trabajo (RPT), Nómina y
Operaciones Bancarias. 
Volumen I - Vigencia: Enero 2026 a Diciembre 2027
Banco Atlas Financiero | Manual Corporativo Expandido v3.0
USO ESTRICTAMENTE CONFIDENCIAL
Página 1

--- Vista previa del Chunk 1 (nota cómo se solapa con el final del chunk 0) ---
Documento 1: Manual Corporativo de Recursos
Humanos
1.1 Historia y Filosofía de Expansión
Banco  Atlas  Financiero  S.A.  inició  sus  operaciones  como  una  caja  de  crédito  local,  pero
rápidamente escaló hacia un modelo de banca múltiple impulsado por la ciencia de datos. Nuestra
misión es empoderar financieramente a los ciudadanos m

In [5]:
!pip install sentence-transformers

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import time

# --- CONFIGURACIÓN 1: Modelo rápido y general ---
print("⏳ Cargando Configuración 1 (puede tardar unos segundos al descargar el modelo)...")
modelo_1 = "all-MiniLM-L6-v2"
embeddings_1 = HuggingFaceEmbeddings(model_name=modelo_1)

# --- CONFIGURACIÓN 2: Modelo multilingüe (ideal para nuestro texto en español) ---
print("⏳ Cargando Configuración 2...")
modelo_2 = "paraphrase-multilingual-MiniLM-L12-v2"
embeddings_2 = HuggingFaceEmbeddings(model_name=modelo_2)

# Vamos a probarlos con nuestro Chunk 0
texto_prueba = chunks[0]

# Medimos y comparamos el Modelo 1
inicio_1 = time.time()
vector_1 = embeddings_1.embed_query(texto_prueba)
fin_1 = time.time()
tiempo_1 = fin_1 - inicio_1

# Medimos y comparamos el Modelo 2
inicio_2 = time.time()
vector_2 = embeddings_2.embed_query(texto_prueba)
fin_2 = time.time()
tiempo_2 = fin_2 - inicio_2

# --- RESULTADOS PARA LA RÚBRICA ---
print("\n✅ --- COMPARACIÓN DE CONFIGURACIONES (RÚBRICA) ---")
print(f"Modelo 1 ({modelo_1}):")
print(f" - Longitud del vector (dimensiones): {len(vector_1)}")
print(f" - Tiempo de ejecución: {tiempo_1:.4f} segundos")

print(f"\nModelo 2 ({modelo_2}):")
print(f" - Longitud del vector (dimensiones): {len(vector_2)}")
print(f" - Tiempo de ejecución: {tiempo_2:.4f} segundos")

print("\n💡 JUSTIFICACIÓN TÉCNICA REVISADA:")
print("1. Ambos modelos generan vectores de 384 dimensiones, manteniendo una huella de memoria idéntica.")
print("2. El primer modelo presenta un tiempo de ejecución mayor debido al 'Cold Start' (inicialización de tensores en memoria).")
print("3. Se elige la Configuración 2 (paraphrase-multilingual) como la ganadora para el sistema RAG, ya que al tener soporte nativo multilingüe, capturará con mayor precisión la semántica de las normativas del Banco Atlas redactadas en español.")

⏳ Cargando Configuración 1 (puede tardar unos segundos al descargar el modelo)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

⏳ Cargando Configuración 2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


✅ --- COMPARACIÓN DE CONFIGURACIONES (RÚBRICA) ---
Modelo 1 (all-MiniLM-L6-v2):
 - Longitud del vector (dimensiones): 384
 - Tiempo de ejecución: 0.0084 segundos

Modelo 2 (paraphrase-multilingual-MiniLM-L12-v2):
 - Longitud del vector (dimensiones): 384
 - Tiempo de ejecución: 0.0124 segundos

💡 JUSTIFICACIÓN TÉCNICA REVISADA:
1. Ambos modelos generan vectores de 384 dimensiones, manteniendo una huella de memoria idéntica.
2. El primer modelo presenta un tiempo de ejecución mayor debido al 'Cold Start' (inicialización de tensores en memoria).
3. Se elige la Configuración 2 (paraphrase-multilingual) como la ganadora para el sistema RAG, ya que al tener soporte nativo multilingüe, capturará con mayor precisión la semántica de las normativas del Banco Atlas redactadas en español.
